# Đối chiếu nội dung parser

Chạy trên toàn bộ HTML pilot đã tải; không gửi request mới. Parser dùng lại cho crawler ở `content_parser.py`. Kết quả chỉ gồm `documents.jsonl` và `review.html` trong `artifacts/parser_review/`.

`extracted_unreviewed` nghĩa là đã trích được text nhưng chưa được người duyệt. Cần xem cả phần bị loại và heading ngoài vùng chọn, không chỉ đọc kết quả trích xuất.

Nếu chưa có HTML pilot, chạy `host_template_audit.ipynb` trước. Có thể chạy parser từ repo root bằng `uv run python content_parser.py`.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / 'dataset').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from scripts.content_parser import run_pilot

PILOT = ROOT / 'artifacts/host_template_audit'
OUTPUT = ROOT / 'artifacts/parser_review'
DOC_ID = 1298551


In [2]:
documents = run_pilot(PILOT, OUTPUT)


{
  "documents": 1820,
  "statuses": {
    "http_error": 73,
    "extracted_unreviewed": 1393,
    "skipped_host": 126,
    "needs_review": 168,
    "encoded_content": 19,
    "fetch_error": 3,
    "access_challenge": 20,
    "homepage": 1,
    "redirected_home": 14,
    "non_html": 1,
    "empty_response": 2
  },
  "warnings": {
    "embedded_content_not_parsed": 10,
    "listing_not_single_article": 8,
    "media_transcript_not_verified": 20,
    "body_requires_decoder": 19,
    "unrendered_publisher_marker": 5,
    "image_content_not_transcribed": 9,
    "pdf_not_parsed": 5,
    "section_previews_linked_details_not_fetched": 20,
    "overview_only_linked_tabs_not_fetched": 17,
    "paywalled_sections": 19,
    "media_only": 2,
    "placeholder_qa": 1,
    "quiz_questions_not_in_server_html": 5,
    "no_answers_in_html": 3,
    "patient_image_unavailable": 2,
    "wiki_tail_requires_review": 40,
    "listing_description_only": 1,
    "chapter_index": 5
  }
}


In [3]:
import pandas as pd
from IPython.display import display

summary = pd.DataFrame([{'host': d['host'], 'status': d['status'], 'doc_id': d['doc_id']} for d in documents])
display(summary.groupby(['host', 'status']).size().unstack(fill_value=0))


status,access_challenge,empty_response,encoded_content,extracted_unreviewed,fetch_error,homepage,http_error,needs_review,non_html,redirected_home,skipped_host
host,,,,,,,,,,,
article.iiyi.com,0,0,0,0,0,0,20,0,0,0,0
ask.39.net,0,0,0,20,0,0,0,0,0,0,0
ask.familydoctor.com.cn,0,0,0,0,0,0,3,0,0,0,0
baby.39.net,0,0,0,20,0,0,0,0,0,0,0
bachmai.gov.vn,0,0,0,20,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...
yanglao.familydoctor.com.cn,0,0,0,0,0,0,2,0,0,0,0
youmed.vn,0,0,0,18,0,0,0,2,0,0,0
ypk.familydoctor.com.cn,0,0,0,0,0,0,1,0,0,0,0


## Kiểm tra các trường hợp dễ sai

Các kiểm tra dưới đây dùng trang nguồn thực tế và một HTML nhỏ để kiểm tra bảo toàn cấu trúc. Chúng không thay thế việc đọc bản nguồn và duyệt từng template.

In [ ]:
from scripts.content_parser import parse_page

by_id = {d['doc_id']: d for d in documents}
assert len(by_id) == len(documents), 'Trùng doc_id'
assert not any(d['status'] == 'parser_error' for d in documents), 'Có parser bị lỗi'
assert not any('block_text_mismatch' in d['warnings'] for d in documents), 'DOM walker bỏ sót hoặc lặp text'

# Một câu đáp thật và một box quảng cáo: chỉ lấy một answer.
assert len(by_id[1298551]['answers']) == 1
assert '帮助网友' not in by_id[1298551]['text']
# Câu hỏi bổ sung được giữ; danh sách gợi ý/bách khoa không lọt vào answer.
assert len(by_id[2291691]['answers']) == 1
assert '症状起因' not in by_id[2291691]['text']
assert 'no_answers_in_html' in by_id[1217776]['warnings']
assert 'no_answers_in_html' in by_id[2910204]['warnings']
assert any(r['kind'] == 'audio' for r in by_id[1029368]['resources'])
assert 'media_transcript_not_verified' in by_id[1029368]['warnings']
assert '您的浏览器不支持 audio 元素' not in by_id[1029368]['text']
# Chương sách ngắn vẫn có nội dung; bảng/heading/list/caption giữ thứ tự.
assert '肺主气' in by_id[893827]['text']
assert '临床检查' in by_id[2516803]['text']
assert '并发症' in by_id[2516803]['text']
assert '参考资料' in by_id[3160915]['text']
assert '相关阅读' not in by_id[3160915]['text']
assert 'NGUỒN THAM KHẢO' in by_id[674694]['text']
assert any(b['role'] == 'references' for b in by_id[763012]['blocks'])
assert '上一篇' not in by_id[836542]['text']
assert 'paywalled_sections' in by_id[2547549]['warnings']
assert '此内容为收费内容' not in by_id[2547549]['text']
assert by_id[2547549]['missing_sections']
assert 'Tải ứng dụng YouMed' not in by_id[764166]['text']
assert by_id[764166]['author_metadata']
assert '1000余本中医古籍' not in by_id[893827]['text']
assert 'Tin liên quan' not in by_id[81756]['text']
assert 'TIN LIÊN QUAN' not in by_id[64780]['text']
assert '白血病 知识' not in by_id[2509930]['text']
assert '症状' in by_id[2509930]['text']
assert '治疗' in by_id[2509930]['text']
assert by_id[91333]['status'] == 'encoded_content'

record = {'doc_id': 0, 'host': 'www.cnkang.com', 'url': 'https://www.cnkang.com/sample', 'state': 'html_response'}
raw = '<div id="endText"><h2>Điều trị</h2><p>Không <a href="/x">ngừng thuốc</a>.</p><ul><li>Liều 5 mg</li><li style="display:none">Bổ sung có ích</li></ul><table><tr><th>Tuổi</th><th>Liều</th></tr><tr><td>10</td><td>5 mg</td></tr></table><figure><img src="/chart.png"><figcaption>Bảng theo dõi</figcaption></figure><div class="related-news">Tin gợi ý</div></div>'.encode()
parsed = parse_page(record, raw)
assert 'Không ngừng thuốc.' in parsed['text']
assert 'Bổ sung có ích' in parsed['text']
assert 'Tuổi | Liều' in parsed['text']
assert 'Bảng theo dõi' in parsed['text']
assert 'Tin gợi ý' not in parsed['text']
assert 'block_text_mismatch' not in parsed['warnings']
# Không xóa mô tả bổ sung chỉ vì nó có lặp lại câu hỏi.
record = {'doc_id': 0, 'host': 'ask.39.net', 'url': 'https://ask.39.net/question/example.html', 'state': 'html_response'}
raw = '<h1 class="ask_tit">Ho</h1><div class="ask_hid"><p class="txt_ms">Ho</p><p class="txt_bc">补充说明：<span>Ho</span><span>Trẻ 10 tuổi, ho 3 ngày</span></p></div><div class="selected"><div class="sele_all"><p class="sele_txt">Câu trả lời</p></div></div>'.encode()
parsed = parse_page(record, raw)
assert 'Trẻ 10 tuổi, ho 3 ngày' in parsed['text']
print('Các kiểm tra bảo toàn nội dung và trường hợp hỏi–đáp đã qua.')


Các kiểm tra bảo toàn nội dung và trường hợp hỏi–đáp đã qua.


## Xem một tài liệu

Đổi `DOC_ID` ở cell cấu hình để xem trang khác. Bản `review.html` cho phép lọc tất cả mẫu theo host, ID hoặc trạng thái.

In [ ]:
import html
from IPython.display import HTML, FileLink

d = by_id[DOC_ID]
print(d['url'])
print('Title:', d['title'])
print('Status:', d['status'], '| Warnings:', ', '.join(d['warnings']))
source = html.escape(d['selected_source_text'])
extracted = html.escape('\n\n'.join(f"[{b['role']}/{b['kind']}] {b['text']}" for b in d['blocks']))
display(HTML(f'<div style="display:grid;grid-template-columns:1fr 1fr;gap:20px"><div><h3>Vùng nguồn</h3><pre style="white-space:pre-wrap">{source}</pre></div><div><h3>Kết quả</h3><pre style="white-space:pre-wrap">{extracted}</pre></div></div>'))
display(pd.DataFrame(d['removed']))
display(pd.DataFrame(d['outside_headings']))
display(pd.DataFrame(d.get('outside_paragraphs', [])))
display(pd.DataFrame(d['resources']))
display(FileLink(str(OUTPUT / 'review.html')))

# Toàn bộ text gốc để kiểm tra cả những vùng chưa được chọn.
import gzip
from bs4 import BeautifulSoup
from scripts.content_parser import decode

if d.get('raw_path'):
    raw = gzip.decompress((PILOT / d['raw_path']).read_bytes())
    full_source = BeautifulSoup(decode(raw)[0], 'html.parser').get_text(' ', strip=True)
    display(HTML('<details><summary>Toàn bộ text của trang gốc</summary><pre style="white-space:pre-wrap">' + html.escape(full_source) + '</pre></details>'))


http://www.120ask.com/question/541841.htm
Title: 为何3个多月的小孩总是放屁而且还很臭.
Status: extracted_unreviewed | Warnings: 


""


,text,path
0,为何3个多月的小孩总是放屁而且还很臭.,h1#d_askH1
1,其他类似问题,div#b_tabbox > span:nth-of-type(1) > div:nth-o...
2,回复区,div#body_main > div:nth-of-type(5) > div:nth-o...
3,相关文章,div#body_main > div:nth-of-type(5) > div:nth-o...


,text,path
0,有问必答 内科 外科 儿科 药品 遗传 美容 检查 减肥 中医科 五官科 传染科 体检科 妇...,html:nth-of-type(1) > body:nth-of-type(1) > di...


""


/home/toska/code/personal/MedXIR/artifacts/parser_review/review.html